In [0]:
# List files in volume 
VOLUME_PATH = "/Volumes/marathos/default/raw"
spark.sql(f"LIST '{VOLUME_PATH}'").display()

In [0]:
# Load bronze table
df = spark.sql("FROM marathos.bronze.raw_supply_chain")
df.display()

In [0]:
df.count()

In [0]:
len(df.columns)

In [0]:
df.printSchema()

### Schema insigts 

* Athlete year of birht: was inferred as a double but should be integer. Probably caused by null values. 
* Athelte average speed: was inferred as a string but should be double. Now it suggests non-numeric values are present and that will have to be investigated in Silver.
* Athelte performance: is a string and that might need to be corrected to a timestamp but it is possible that might not work and will have to be a double and calculated.

This will be done in silver EDA/Cleaning

In [0]:
df.describe().display()

* athlete year of birth seems to have som input error considering there is someone born 1193 and somone born 2021.
* Athlete performance contains times that cover days wich should be dropped in silver.
* There are actually data from 1798 wich is very cool! 
* Mean/stddv are meaningless for club and distance sins they are strings.


In [0]:
from pyspark.sql.functions import col, sum as spark_sum

# Count the number of null values in each column
null_counts = df.select(
    [spark_sum(col(column).isNull().cast("int")).alias(column) for column in df.columns]
)

# Convert the result to a dictionary so that we can loop through it
null_counts = null_counts.collect()[0].asDict()

# Only keep volumes with null values
[(column, nulls) for column, nulls in null_counts.items() if nulls > 0]

### Null count reflection 

* Athlete club is high but this is not unlikley due to club membership probably being optional. 
* Athlete performance country and gender are so smal they will probably be droped in silver
* Athlete average speed might is something that we can calculate depending on the other data for that person, Meaning if we have there time and distnace, will otherwise be dropped.

In [0]:
# how many unique events are their?

df.select("Event name").distinct().count()

In [0]:
# What types of distances exist? km, mi, h, d?
df.withColumn(
    "distance_type",
    col("Event distance/length").rlike(r"(?i)km").cast("string")
).groupBy("Event distance/length").count().orderBy(col("count").desc()).limit(20).display()

In [0]:
# Gender distribution
df.groupBy("Athlete gender").count().orderBy(col("count").desc()).display()

### Insights: gender distrubution 

* Majority of athletes are male (M) 
* Small number of x (non-binary) athletes presented
* 7 null values - will be dropped in silver

In [0]:
import plotly.express as px

age_dist = (
    df.groupBy("Athlete age category")
    .count()
    .filter(col("Athlete age category").isNotNull())
    .orderBy("Athlete age category")
    .toPandas()
)

fig = px.histogram(
    age_dist,
    x = "Athlete age category",
    y = "count",
    title = "Age distribution of athletes",
    labels = {"Athlete age category": "Age category", "count": "Number of athletes"},
)

fig.show()

# Country

In [0]:
# Check country values and their frequency
country_dist = (
    df.groupBy("Athlete country")
    .count()
    .filter(col("Athlete country").isNotNull())
    .orderBy(col("count").desc())
)

country_dist.display()

In [0]:
country_dist = (
    df.groupBy("Athlete country")
    .count()
    .filter(col("Athlete country").isNotNull())
    .toPandas()
)

country_dist["country_normalized"] = (
    country_dist["Athlete country"].str.upper().str.strip()
)

country_dist[
    country_dist["country_normalized"].duplicated(keep=False)
].sort_values("country_normalized")

In [0]:
# Check how many records have an unknown country code
df.filter(
    col("Athlete country") == "XXX"
).count()

In [0]:
country_dist = (
    df.groupBy("Athlete country")
    .count()
    .filter(col("Athlete country").isNotNull())
    .toPandas()
    .sort_values("count", ascending=False)
    .head(15)
)

fig = px.bar(
    country_dist,
    x = "Athlete country",
    y = "count",
    title = "Top 15 most represented countries",
    labels = {"Athlete country": "Country", "count": "Number of athletes"},
    text = "count",
)

fig.show()

### Insights: Country 
* USA dominates with 1,389,960 results 
* European countries are well represented 
* There are 208 distinct countries represented 
* XXX Country code exists - means unknown nationality will be dropped in silver. 

In [0]:
# Athlete average spped displays what looks like a times and not a speed. 
# May be possible to recalculate some of the speeds using performance and distance

df.filter(
    ~col("Athlete average speed").rlike(r"^\d+\.?\d*$") & 
    col("Athlete average speed").isNotNull()
).select("Athlete average speed", "Athlete performance", "Event distance/length").distinct().display()

In [0]:
# Shows dates that dosent match the format dd.mm.yyyy insted some are start and finsih dates while others are single digit 
df.filter(
    ~col("Event dates").rlike(r"^\d{2}\.\d{2}\.\d{4}$")
).select("Event dates").distinct().limit(20).display()

In [0]:
# How many stage/multi-stage events exist?
stage_count = df.filter(
    col("Event distance/length").contains("/")
).count()
print(f"Stage events: {stage_count:,}")

In [0]:
spark.table("marathos.bronze.raw_supply_chain") \
    .filter(col("`Athlete age category`").rlike("(?i)^w")) \
    .select("`Athlete age category`").distinct().show()

In [0]:
spark.table("marathos.bronze.raw_supply_chain") \
    .filter(col("`Athlete age category`").rlike("(?i)^m")) \
    .select("`Athlete age category`").distinct().orderBy("Athlete age category").show(30)

In [0]:
# this shows 4 male runners that has ,ade it in to the female athlete_gender
# These rows will be dropped (Probably due to manual data entry)

mismatches = spark.sql("""
    SELECT athlete_age_category, athlete_gender, COUNT(*) AS cnt
    FROM marathos.silver.cleaned_marathos_2
    WHERE (athlete_age_category LIKE 'F%' AND athlete_gender = 'M')
       OR (athlete_age_category LIKE 'M%' AND athlete_gender = 'F')
    GROUP BY athlete_age_category, athlete_gender
    ORDER BY cnt DESC
""")
mismatches.display()

### Summary
# To do in silver layer

* rename all columns to snake_case 
* Change datatypes:
    - athelete_year_of_birth: double -> int, null out values outside 1900-2010
    - athlete_average_speed: string -> double, reluctante from scratch
* Drop an fill null vlaues per column analysis above 
* Clean event_dates - extract start date to match dd.mm.yy
* Drop stage events (contains "/") and day events (unit "d")
* Drop multi-day performances (e.g "3d 02:03:00 h")
* Drop XXX athletes
* Athlete_country contains "SWE" and "swe" this mmight duplicate and create problems in the future 
* Recalculate athelte_average_speed and filter to 0.5-20 km/h range
* Generate surrogate IDs using sha2 
* 4 male competitors in female athlete_gender category will be dropped